In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
import warnings
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

warnings.filterwarnings("ignore")
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
login(hf_token)

!pip install wandb -q
import wandb
wandb_token = secrets.get_secret("WANDB")
wandb.login(key=wandb_token)

import pyarrow as pa
import pyarrow.parquet as pq

# For Colab setup
'''from google.colab import userdata
from huggingface_hub import login, snapshot_download

login(userdata.get('HF_TOKEN'))  # login first, before snapshot_download'''

'''try:
    snapshot_download(
        repo_id="alexdimmock/wav2vec2-basque-10h",
        local_dir="/content/wav2vec2-basque-10h"
    )
    print("Checkpoint restored from Hub")
except Exception as e:
    print(f"No checkpoint found, starting fresh: {e}")'''

In [ ]:
!pip install transformers datasets evaluate jiwer -q

from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor, TrainingArguments, Trainer
from datasets import load_dataset, Dataset, Audio, get_dataset_config_names, get_dataset_split_names
from dataclasses import dataclass
from evaluate import load as load_metric
import os, torch
import unicodedata
import re
import numpy as np

# CPU Fallback
'''os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
'''
# Load pretrained characters from basque fine-tune for token characters
processor = Wav2Vec2Processor.from_pretrained("stefan-it/wav2vec2-large-xlsr-53-basque")

print("\nBasque token vocabulary:")
print(processor.tokenizer.get_vocab())

# Load base facebook model to fine-tune
ssl_model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-large-xlsr-53", vocab_size=len(processor.tokenizer))

ssl_model.freeze_feature_encoder()

# Text cleaner
def clean_text(t):
    t = t.lower()
    t = unicodedata.normalize("NFKC", t)
    t = re.sub(r"[^\w\sñíáéóúü]", "", t)  # keep letters only
    t = re.sub(r"\s+", " ", t).strip()

    return t

In [ ]:
def preprocess(sample):
    '''
    Preprocess: takes a sample, separates the audio, sampling rate, input and labels
    Args: sample
    Returns: dict with input values and labels
    '''
    audio = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

    inputs = processor(audio, sampling_rate=sr)
    
    text = clean_text(sample["sentence"])
    labels = processor(text=[text]).input_ids[0]
    
    return {
        "input_values": inputs.input_values[0],
        "labels": labels
    }

In [ ]:
'''### TO DETERMINE NUMBER OF SAMPLES REQUIRED FOR NUMBER OF HOURS

ds = load_dataset("HiTZ/composite_corpus_eu_v2.1", split="train", streaming=True)

total_duration = 0
count = 0
for sample in ds:
    total_duration += len(sample["audio"]["array"]) / sample["audio"]["sampling_rate"]
    count += 1
    if count % 500 == 0:
        print(f"{count} samples, {total_duration/3600:.2f} hours so far")
    if total_duration >= 50 * 3600:  # stop at 10 hours
        break

print(f"50 hours reached at sample {count}")'''

In [ ]:
'''# Load 10h dataset
raw = load_dataset("HiTZ/composite_corpus_eu_v2.1", split="train", streaming=True).take(6000)
train_list = [preprocess(s) for s in raw]
cv_train_10h = Dataset.from_list(train_list)'''

'''# Load 50h dataset
ds_50 = load_dataset("HiTZ/composite_corpus_eu_v2.1", split="train", streaming=True).take(30500)
train_50h_list = [preprocess(s) for s in ds_50]
train_50h_list = [s for s in train_50h_list if len(s["input_values"]) < 320000]  # ← filter here
cv_train_50h = Dataset.from_list(train_50h_list)'''

ds_50 = load_dataset("HiTZ/composite_corpus_eu_v2.1", split="train", streaming=True).take(30500)

writer = None
chunk_input = []
chunk_labels = []
chunk_size = 500

for i, s in enumerate(ds_50):
    p = preprocess(s)
    if len(p["input_values"]) < 320000:
        chunk_input.append(p["input_values"].tolist())
        chunk_labels.append(p["labels"])
    
    if len(chunk_input) == chunk_size:
        table = pa.table({"input_values": chunk_input, "labels": chunk_labels})
        if writer is None:
            writer = pq.ParquetWriter("/kaggle/working/train_50h.parquet", table.schema)
        writer.write_table(table)
        chunk_input, chunk_labels = [], []
        print(f"Written {i} samples", flush=True)

# write remaining
if chunk_input:
    table = pa.table({"input_values": chunk_input, "labels": chunk_labels})
    if writer is None:
        writer = pq.ParquetWriter("/kaggle/working/train_50h.parquet", table.schema)
    writer.write_table(table)

writer.close()

# === DISK CLEANUP ===
import gc, shutil, os

# Free the Python objects from memory
del ds_50, chunk_input, chunk_labels, table
gc.collect()

# Remove the output_dir from any previous run (old checkpoints eating space)
if os.path.exists("/kaggle/working/wav2vec2-basque-50h"):
    shutil.rmtree("/kaggle/working/wav2vec2-basque-50h")
    print("Cleared old output dir")
# =====================

cv_train_50h = Dataset.from_parquet("/kaggle/working/train_50h.parquet")
print(f"Dataset size: {len(cv_train_50h)}")

os.remove("/kaggle/working/train_50h.parquet")
print("Deleted parquet file to free disk space")

# Validation set
cv_dev_raw = load_dataset("HiTZ/composite_corpus_eu_v2.1", split="dev_cv", streaming=True).take(500)
cv_dev = []
for i, sample in enumerate(cv_dev_raw.map(preprocess)):
    cv_dev.append(sample)
    if i % 50 == 0:
        print(f"Loaded {i} val samples")
print(f"Done: {len(cv_dev)} val samples")

@dataclass
class CTCDataCollator:
    processor: Wav2Vec2Processor

    def __call__(self, features):
        # Pad input_values
        input_features = [
            {"input_values": f["input_values"]}
            for f in features]
            
        batch = self.processor.pad(input_features, padding=True, return_tensors="pt")

        # Pad labels with PAD token
        label_features = [f["labels"] for f in features]
        labels_batch = self.processor.tokenizer.pad(
            {"input_ids": label_features}, padding=True, return_tensors="pt"
        )
        # Replace PAD token id with -100 so loss ignores it
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["input_ids"] == self.processor.tokenizer.pad_token_id, -100
        )
        batch["labels"] = labels
        return batch

data_collator = CTCDataCollator(processor=processor)

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir="/kaggle/working/wav2vec2-basque-50h",
    report_to="wandb",
    run_name="basque-50h-run5",
    per_device_train_batch_size=2,
    eval_strategy="epoch",
    bf16=True,
    learning_rate=5e-4,
    warmup_steps=200,
    logging_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=5,
    save_total_limit=2,
    push_to_hub=True,
    hub_model_id="alexdimmock/wav2vec2-basque-50h",
    hub_strategy="checkpoint",
    hub_token=hf_token
    )

wer_metric = load_metric("wer")

def compute_metrics(pred):
    logits = pred.predictions
    pred_ids = np.argmax(logits, axis=-1)

    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids, group_tokens=True)
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}
    
trainer = Trainer(
    model=ssl_model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=cv_train_50h,
    eval_dataset=cv_dev,
    processing_class=processor.feature_extractor
)

trainer.train()

'''# Exit file
os._exit(0)'''

In [ ]:
'''### DEBUG ON SMALL OVERFITTED DATASET

from datasets import load_dataset, Dataset

raw = load_dataset("HiTZ/composite_corpus_eu_v2.1", split="train", streaming=True).take(500)
train_list = [preprocess(s) for s in raw]
cv_train_10h = Dataset.from_list(train_list)

# Quick sanity check: print what labels actually look like now
print("labels sample:", cv_train_10h[0]["labels"])
print("decoded:", processor.tokenizer.decode(cv_train_10h[0]["labels"]))

training_args = TrainingArguments(
    output_dir="./debug-overfit",
    per_device_train_batch_size=2,
    learning_rate=4e-4,   # was 1e-4
    max_steps=5000,       # was 200
    logging_steps=50,
    eval_strategy="no",
    save_strategy="steps",
    bf16=True,
)

trainer = Trainer(
    model=ssl_model,
    args=training_args,
    train_dataset=cv_train_10h,
    data_collator=data_collator,
    processing_class=processor,
)

trainer.train()

# Check a prediction after training
import torch, numpy as np
sample = cv_train_10h[10]
input_tensor = torch.tensor([sample["input_values"]]).to(ssl_model.device)
with torch.no_grad():
    logits = ssl_model(input_tensor).logits
pred_ids = logits.argmax(-1)
print("PRED:", processor.batch_decode(pred_ids)[0])
print("TRUE:", processor.tokenizer.decode(sample["labels"]))'''



'''try wer function on debug code if the above is fucked'''

In [ ]:
import os
for root, dirs, files in os.walk("/kaggle/working/last-checkpoint"):
    for f in files:
        print(os.path.join(root, f))